# Logging Pipeline Metrics with MLflow

This notebook demonstrates how to use the `MLFlowLoggerWrapper` to automatically log pipeline metadata, metrics, and artifacts to MLflow.

The wrapper sits around any `PipelineRunner` and intercepts the `train()` / `evaluate()` lifecycle:

- **On `train()`** -- opens an MLflow run, logs pipeline metadata as tags, saves the full pipeline config JSON as an artifact, renders an interactive HTML graph of the DAG, delegates to the runner's `train()`, then saves the pipeline's trained parameters (learned weights) as an artifact.
- **On `evaluate()`** -- resumes the same MLflow run, delegates to the runner's `evaluate()`, and logs all metric node outputs as MLflow metrics (R2, MSE, etc.).

The wrapper config supports **environment variables** (prefix `MLFLOW_`), so in a team setting you can set `MLFLOW_TRACKING_URI` once and every notebook/script picks it up automatically.

## 1. Build a Pipeline

We reuse the same pipeline from the `pipeline.ipynb` example: CSV sources, transforms, a RandomForestRegressor, and R2 + MSE metrics.

In [1]:
import sys

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig

configure(level="INFO", stream=sys.stdout, fmt="text")

# --- Data sources ---
csv_config_class = NODE_REGISTRY.get_node_config_class("TabularCSVFetcher")

data_csv_config = csv_config_class()
data_csv_config.running_config.csv_path = "../data/fake_batch.csv"
data_csv_config.running_config.context_path = "../data/fake_batch_context.json"

target_csv_config = csv_config_class()
target_csv_config.running_config.csv_path = "../data/fake_target_df.csv"
target_csv_config.running_config.context_path = "../data/fake_target_context.json"

# --- Transform configs ---
missing_filter_config_class = NODE_REGISTRY.get_node_config_class("MissingRateFilter")
data_missing_filter_config = missing_filter_config_class()
target_missing_filter_config = missing_filter_config_class()
target_missing_filter_config.in_ports["input"].mode = ["training", "evaluation"]
target_missing_filter_config.hyperparameters.threshold = 0.0

# --- Assemble pipeline ---
nodes = {
    "data_source": ("TabularCSVFetcher", data_csv_config),
    "target_source": ("TabularCSVFetcher", target_csv_config),
    "data_missing_filter": ("MissingRateFilter", data_missing_filter_config),
    "category_splitter": ("DataCategoryFilter", NODE_REGISTRY.get_node_config_class("DataCategoryFilter")()),
    "one_hot_encoder": ("OneHotEncoding", NODE_REGISTRY.get_node_config_class("OneHotEncoding")()),
    "target_missing_filter": ("MissingRateFilter", target_missing_filter_config),
    "iqr_filter": ("IQROutlierFilter", NODE_REGISTRY.get_node_config_class("IQROutlierFilter")()),
    "variance_threshold_filter": ("VarianceFilter", NODE_REGISTRY.get_node_config_class("VarianceFilter")()),
    "standard_scaler": ("StandardScaler", NODE_REGISTRY.get_node_config_class("StandardScaler")()),
    "feature_concatenate": ("FeatureConcatenate", NODE_REGISTRY.get_node_config_class("FeatureConcatenate")()),
    "random_forest_regressor": ("RandomForestRegressor", NODE_REGISTRY.get_node_config_class("RandomForestRegressor")()),
    "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
    "mse": ("MSE", NODE_REGISTRY.get_node_config_class("MSE")()),
    "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

edges = [
    Edge(source="data_source", target="data_missing_filter", ports_map=[("output", "input")]),
    Edge(source="data_missing_filter", target="category_splitter", ports_map=[("output", "input")]),
    Edge(source="one_hot_encoder", target="feature_concatenate", ports_map=[("output", "input_2")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("numerical", "input")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("categorical", "sliced")]),
    Edge(source="target_source", target="iqr_filter", ports_map=[("output", "target")]),
    Edge(source="iqr_filter", target="variance_threshold_filter", ports_map=[("output", "input")]),
    Edge(source="iqr_filter", target="one_hot_encoder", ports_map=[("sliced", "input")]),
    Edge(source="variance_threshold_filter", target="standard_scaler", ports_map=[("output", "input")]),
    Edge(source="standard_scaler", target="feature_concatenate", ports_map=[("output", "input_1")]),
    Edge(source="feature_concatenate", target="random_forest_regressor", ports_map=[("output", "X")]),
    Edge(source="iqr_filter", target="target_missing_filter", ports_map=[("target", "input")]),
    Edge(source="target_missing_filter", target="random_forest_regressor", ports_map=[("output", "y")]),
    Edge(source="random_forest_regressor", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="r2", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="mse", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="mse", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="sink", ports_map=[("pred", "dump")]),
]

pipe = Pipeline(config=PipelineConfig(nodes=nodes, edges=edges, name="MLflow Demo Pipeline"))
pipe.compile()
print("Pipeline compiled.")

NodeRegistry with 0 registered nodes: []
Successfully registered nodes from module: tsut.components.nodes.data_sources._register
Successfully registered nodes from module: tsut.components.nodes.metrics.classification._register
Successfully registered nodes from module: tsut.components.nodes.metrics.regression._register
Successfully registered nodes from module: tsut.components.nodes.models._register
Successfully registered nodes from module: tsut.components.nodes.sinks._register
Successfully registered nodes from module: tsut.components.nodes.transforms.encodings._register
Successfully registered nodes from module: tsut.components.nodes.transforms.feature_selection._register
Successfully registered nodes from module: tsut.components.nodes.transforms.filters._register
Successfully registered nodes from module: tsut.components.nodes.transforms.imputations._register
Successfully registered nodes from module: tsut.components.nodes.transforms.operations._register
Successfully registered nod

## 2. Wrap the Runner with MLflow Logging

Instead of calling `SmartRunner` directly, we wrap it with `MLFlowLoggerWrapper`. The wrapper config controls what gets logged.

Key options:
- **`experiment_name`** -- MLflow experiment to log under
- **`run_name`** -- human-readable run label
- **`log_pipeline_params`** -- save the pipeline's trained parameters (learned weights) as a JSON artifact after training
- **`log_pipeline_config`** -- save the full pipeline config JSON as an artifact
- **`log_pipeline_graph`** -- save an interactive HTML render of the pipeline DAG as an artifact
- **`log_train_time`** -- record training wall-clock time as a metric
- **`tags`** -- custom key-value tags attached to the run

All string/bool fields can also be set via environment variables with the `MLFLOW_` prefix (e.g. `MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT_NAME`).

In [ ]:
from tsut.core.pipeline.runners.smart_runner import SmartRunner
from tsut.components.pipe_runner_wrappers.mlflow_logger import (
    MLFlowLoggerWrapper,
    MLFlowLoggerWrapperConfig,
)

runner = SmartRunner(pipe)

mlflow_runner = MLFlowLoggerWrapper(
    runner,
    config=MLFlowLoggerWrapperConfig(
        experiment_name="tsut_demo",
        run_name="random_forest_baseline",
        log_pipeline_params=False,
        log_pipeline_config=True,
        log_pipeline_graph=True,
        log_train_time=True,
        tags={
            "author": "notebook_demo",
            "dataset": "fake_batch",
        },
    ),
)

## 3. Train and Evaluate

Calling `train()` on the wrapper:
1. Opens an MLflow run
2. Logs pipeline metadata as tags (name, version, node list)
3. Logs custom tags (`author`, `dataset`)
4. Saves `pipeline_config.json` as an artifact
5. Saves `pipeline_graph.html` (interactive DAG visualisation) as an artifact
6. Delegates to the underlying runner's `train()`
7. Logs `train_duration_s`
8. Saves the pipeline's trained parameters (learned weights) as a JSON artifact

Calling `evaluate()` afterwards:
1. Resumes the **same** MLflow run
2. Delegates to the runner's `evaluate()`
3. Logs each metric node's scalar output (e.g. `r2`, `mse`)
4. Logs `evaluate_duration_s`

In [5]:
mlflow_runner._runner.pipeline.get_params()

{'data_missing_filter': {'columns_to_keep': ['output_batch_id',
   'total_pct',
   'ingredient_count',
   'pct_entropy',
   'total_pct_CHG',
   'total_pct_Co',
   'total_pct_WC',
   'top1_element',
   'top1_pct',
   'top2_element',
   'top2_pct',
   'mean_CHG_TANBC60%',
   'mean_CHG_CARBON_TOTAL_LU',
   'mean_CHG_PARAFFIN_RATE',
   'mean_CHG_WC_RATE',
   'mean_CHG_CO_RATE',
   'mean_CHG_VC_RATE',
   'mean_CHG_CR2N_RATE',
   'mean_CHG_NI_RATE',
   'mean_CHG_MO2C_RATE',
   'mean_CHG_TANBC60_RATE',
   'mean_CHG_TIC_RATE',
   'mean_CHG_%CTOT_LU',
   'mean_CHG_CO%',
   'mean_CHG_VC%',
   'mean_CHG_CR2N%',
   'mean_CHG_TIC%',
   'mean_CHG_WC%',
   'mean_CHG_4PS (G)',
   'mean_CHG_LOSS-VAC',
   'mean_CHG_4PS (S)',
   'mean_CHG_HC',
   'mean_CHG_CONTR 25',
   'mean_CHG_CONTR 75',
   'mean_CHG_CONTR125',
   'mean_CHG_MO2C%',
   'mean_CHG_NI%',
   'mean_Co_FS_AS_V',
   'mean_Co_%O_V',
   'mean_VC_FS_AS_V',
   'mean_VC_%CTOT_V',
   'mean_VC_%CFREE_V',
   'mean_VC_%O_V',
   'mean_VC_%CTOT_LU',
   

In [4]:
# Train -- this creates the MLflow run and logs params + artifacts
mlflow_runner.train()

11:07:51 INFO     [tsut.runner.smart] Phase training start  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  pipeline_phase=training  phase_status=start
11:07:51 INFO     [tsut.runner.smart] Node executed: data_source  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_name=data_source  pipeline_phase=training  duration_ms=5.934834014624357  node_type=source
11:07:51 INFO     [tsut.runner.smart] Node executed: data_missing_filter  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_name=data_missing_filter  pipeline_phase=training  duration_ms=1.5249999705702066  node_type=transform
11:07:51 INFO     [tsut.runner.smart] Node executed: category_splitter  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_name=category_splitter  pipeline_phase=training  duration_ms=0.7023749640211463  node_type=transform
11:07:51 INFO     [tsut.runner.smart] Node executed: target_source  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_n

TypeError: Object of type DecisionTreeRegressor is not JSON serializable

In [ ]:
# Evaluate -- resumes the same run and logs metrics
metrics = mlflow_runner.evaluate()

print("Run ID:", mlflow_runner.run_id)
print("Metrics logged to MLflow:")
for name, value in metrics.items():
    print(f"  {name}: {value[value.keys().__iter__().__next__()].to_numpy()[0][0, 0]:.4f}")

10:43:29 INFO     [tsut.runner.smart] Phase evaluation start  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  pipeline_phase=evaluation  phase_status=start
10:43:29 INFO     [tsut.runner.smart] Node executed: data_source  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_name=data_source  pipeline_phase=evaluation  duration_ms=0.11733395513147116  node_type=source
10:43:29 INFO     [tsut.runner.smart] Node executed: data_missing_filter  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_name=data_missing_filter  pipeline_phase=evaluation  duration_ms=0.7877920288592577  node_type=transform
10:43:29 INFO     [tsut.runner.smart] Node executed: category_splitter  pipeline_name=MLflow Demo Pipeline  pipeline_version=0.1.0  node_name=category_splitter  pipeline_phase=evaluation  duration_ms=0.5513749783858657  node_type=transform
10:43:29 INFO     [tsut.runner.smart] Node executed: target_source  pipeline_name=MLflow Demo Pipeline  pipeline_version=0

## 4. Inspect the Run in MLflow

You can view the logged run using the MLflow UI. Start it from the terminal:

```bash
mlflow ui --port 5000
```

Then open http://localhost:5000 in your browser. You should see:
- **Metrics** tab: `r2`, `mse`, `train_duration_s`, `evaluate_duration_s`
- **Artifacts** tab: `pipeline_config.json`, `pipeline_graph.html`, and the trained parameters JSON
- **Tags**: `pipeline_name`, `pipeline_version`, `pipeline_nodes`, `author`, `dataset`

You can also query the run programmatically:

In [ ]:
import mlflow

run = mlflow.get_run(mlflow_runner.run_id)

print("=== Tags ===")
for k, v in sorted(run.data.tags.items()):
    if not k.startswith("mlflow."):  # skip internal mlflow tags
        print(f"  {k}: {v}")

print("\n=== Metrics ===")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k}: {v:.4f}")

print("\n=== Artifacts ===")
client = mlflow.tracking.MlflowClient()
for artifact in client.list_artifacts(mlflow_runner.run_id):
    print(f"  {artifact.path}")

=== Tags ===
  author: notebook_demo
  dataset: fake_batch
  pipeline_name: MLflow Demo Pipeline
  pipeline_nodes: data_source, target_source, data_missing_filter, category_splitter, one_hot_encoder, target_missing_filter, iqr_filter, variance_threshold_filter, standard_scaler, feature_concatenate, random_forest_regressor, r2, mse, sink
  pipeline_version: 0.1.0

=== Metrics ===
  evaluate_duration_s: 0.0392
  train_duration_s: 0.3265

=== Artifacts ===
  MLflow Demo Pipeline_v0.1.0_params.json
  pipeline_config.json
  pipeline_graph.html


## 5. Environment Variable Configuration

In a team or CI/CD setting, you can configure the wrapper entirely through environment variables instead of passing values in code. This is especially useful for the tracking URI:

```bash
export MLFLOW_TRACKING_URI="http://mlflow.internal:5000"
export MLFLOW_EXPERIMENT_NAME="production_pipelines"
```

Then in code you just need:

```python
mlflow_runner = MLFlowLoggerWrapper(runner)  # picks up env vars automatically
```

The full list of supported environment variables:

| Env Variable | Config Field | Default |
|---|---|---|
| `MLFLOW_TRACKING_URI` | `tracking_uri` | `None` (local) |
| `MLFLOW_EXPERIMENT_NAME` | `experiment_name` | `"tsut"` |
| `MLFLOW_RUN_NAME` | `run_name` | `None` |
| `MLFLOW_NESTED` | `nested` | `False` |
| `MLFLOW_LOG_PIPELINE_PARAMS` | `log_pipeline_params` | `True` |
| `MLFLOW_LOG_PIPELINE_CONFIG` | `log_pipeline_config` | `True` |
| `MLFLOW_LOG_PIPELINE_GRAPH` | `log_pipeline_graph` | `True` |
| `MLFLOW_LOG_TRAIN_TIME` | `log_train_time` | `False` |